# Simulation/Rating matrix test

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
workspace_root = next((p for p in [cwd, *cwd.parents] if (p / 'core').is_dir()), None)
if workspace_root is None:
    raise RuntimeError("Nie znaleziono katalogu gĹ‚Ăłwnego repozytorium.")
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

from core.geometry.tube import BareTube
from core.geometry.bundle import TubeBundle
from core.properties import (
    DryAirPropertyProvider,
    GasMixturePropertyProvider,
    GasMixtureSpec,
)
from core.models.bare_tube import BareTubeHeatExchanger
from core.models.simulation import HXSideInput
from core.models.heat_balance import BalanceSideSpec


In [ ]:
def c_to_k(value_c):
    return None if value_c is None else value_c + 273.15

def kgh_to_kgs(value):
    return None if value is None else value / 3600.0

wet_gas_components = {
    'N2': 0.6627,
    'O2': 0.1761,
    'H2O': 0.1612,
}
wet_gas_spec = GasMixtureSpec(
    components=wet_gas_components, basis='mole', backend='HEOS', imposed_phase='gas',
)

media = {
    'dry_air': {
        'label': 'suche powietrze',
        'provider': DryAirPropertyProvider(),
    },
    'wet_gas_mixture': {
        'label': 'mieszanina N2/O2/H2O (gaz)',
        'provider': GasMixturePropertyProvider(wet_gas_spec),
    },
}

tube = BareTube(D_i=34.8e-3, D_o=38.1e-3, length_total=2.2, length_effective=2.15, wall_k=50.0)
euler_provider = 'gaddis_gnielinski'
# euler_provider = 'zukauskas'
bundle = TubeBundle(
    tube=tube, n_rows=20, n_tubes_per_row=40,
    pitch_transverse=48e-3, pitch_longitudinal=48e-3,
    layout='inline', n_passes_tube=1, flow_arrangement='crossflow',
)
hx = BareTubeHeatExchanger(bundle)


## Cases Matrix

Use `mode='simulate'` to calculate achievable duty and outlet temperatures. `overdesign` derates the effective conductance using `UA_eff = UA / (1 + overdesign)`.

Use `mode='rate'` to evaluate a specified heat balance. Provide `Q_kW`, `effectiveness`, or one complete side with an outlet temperature. Use `None` for values solved by the model.

Select `dry_air` or `wet_gas_mixture` independently for each side. 

In [ ]:
cases = [

    {
        'case': 'W1dirtySummer', 'mode': 'rate', 'overdesign': None,
        'inside_medium': 'wet_gas_mixture', 
        'inside_m_dot_kg_h': 28_000.0, 'inside_T_in_C': 300.0, 'inside_T_out_C': None,
        'outside_medium': 'dry_air',
        'outside_m_dot_kg_h': 18_000.0, 'outside_T_in_C': 180.0, 'outside_T_out_C': 245.0,
        'Q_kW': None, 'effectiveness': None,
    },
    {
        'case': 'W1dirtyWinter', 'mode': 'rate', 'overdesign': None,
        'inside_medium': 'wet_gas_mixture', 
        'inside_m_dot_kg_h': 28_000.0, 'inside_T_in_C': 280.0, 'inside_T_out_C': None,
        'outside_medium': 'dry_air',
        'outside_m_dot_kg_h': 21_000.0, 'outside_T_in_C': 135.0, 'outside_T_out_C': 207.0,
        'Q_kW': None, 'effectiveness': None,
    },
    {
        'case': 'W1cleanSummer', 'mode': 'rate', 'overdesign': None,
        'inside_medium': 'wet_gas_mixture', 
        'inside_m_dot_kg_h': 28_000.0, 'inside_T_in_C': 275.0, 'inside_T_out_C': 255.3,
        'outside_medium': 'dry_air',
        'outside_m_dot_kg_h': 18_000.0, 'outside_T_in_C': 215.0, 'outside_T_out_C': None,
        'Q_kW': None, 'effectiveness': None,
    },
    {
        'case': 'W1cleanWinter', 'mode': 'rate', 'overdesign': None,
        'inside_medium': 'wet_gas_mixture', 
        'inside_m_dot_kg_h': 28_000.0, 'inside_T_in_C': 250.0, 'inside_T_out_C': 225.5,
        'outside_medium': 'dry_air',
        'outside_m_dot_kg_h': 21_000.0, 'outside_T_in_C': 180.0, 'outside_T_out_C': None,
        'Q_kW': None, 'effectiveness': None,
    },

    {
        'case': 'W2dirtySummer', 'mode': 'rate', 'overdesign': None,
        'inside_medium': 'dry_air', 
        'inside_m_dot_kg_h': 18_000.0, 'inside_T_in_C': 30.0, 'inside_T_out_C': 145.0,
        'outside_medium': 'wet_gas_mixture',
        'outside_m_dot_kg_h': 28_000.0, 'outside_T_in_C': 326.0, 'outside_T_out_C': None,
        'Q_kW': None, 'effectiveness': None,
    },
    {
        'case': 'W2dirtyWinter', 'mode': 'rate', 'overdesign': None,
        'inside_medium': 'dry_air', 
        'inside_m_dot_kg_h': 21_000.0, 'inside_T_in_C': -15.0, 'inside_T_out_C': 102.0,
        'outside_medium': 'wet_gas_mixture',
        'outside_m_dot_kg_h': 28_000.0, 'outside_T_in_C': 312.0, 'outside_T_out_C': None,
        'Q_kW': None, 'effectiveness': None,
    },
    {
        'case': 'W2cleanSummer', 'mode': 'rate', 'overdesign': None,
        'inside_medium': 'dry_air', 
        'inside_m_dot_kg_h': 18_000.0, 'inside_T_in_C': 30.0, 'inside_T_out_C': 163.5,
        'outside_medium': 'wet_gas_mixture',
        'outside_m_dot_kg_h': 28_000.0, 'outside_T_in_C': 315.0, 'outside_T_out_C': None,
        'Q_kW': None, 'effectiveness': None,
    },
    {
        'case': 'W2cleanWinter', 'mode': 'rate', 'overdesign': None,
        'inside_medium': 'dry_air', 
        'inside_m_dot_kg_h': 21_000.0, 'inside_T_in_C': -15.0, 'inside_T_out_C': 118.0,
        'outside_medium': 'wet_gas_mixture',
        'outside_m_dot_kg_h': 28_000.0, 'outside_T_in_C': 295.0, 'outside_T_out_C': None,
        'Q_kW': None, 'effectiveness': None,
    },

    {
        'case': 'W3dirtySummer', 'mode': 'rate', 'overdesign': None,
        'inside_medium': 'wet_gas_mixture', 
        'inside_m_dot_kg_h': 28_000.0, 'inside_T_in_C': 334.0, 'inside_T_out_C': None,
        'outside_medium': 'dry_air',
        'outside_m_dot_kg_h': 18_000.0, 'outside_T_in_C': 30.0, 'outside_T_out_C': 191.0,
        'Q_kW': None, 'effectiveness': None,
    },
    {
        'case': 'W3dirtyWinter', 'mode': 'rate', 'overdesign': None,
        'inside_medium': 'wet_gas_mixture', 
        'inside_m_dot_kg_h': 28_000.0, 'inside_T_in_C': 317.0, 'inside_T_out_C': None,
        'outside_medium': 'dry_air',
        'outside_m_dot_kg_h': 21_000.0, 'outside_T_in_C': -15.0, 'outside_T_out_C': 145.0,
        'Q_kW': None, 'effectiveness': None,
    },
    {
        'case': 'W3cleanSummer', 'mode': 'rate', 'overdesign': None,
        'inside_medium': 'wet_gas_mixture', 
        'inside_m_dot_kg_h': 28_000.0, 'inside_T_in_C': 320.0, 'inside_T_out_C': None,
        'outside_medium': 'dry_air',
        'outside_m_dot_kg_h': 18_000.0, 'outside_T_in_C': 30.0, 'outside_T_out_C': 192.3,
        'Q_kW': None, 'effectiveness': None,
    },
    {
        'case': 'W3cleanWinter', 'mode': 'rate', 'overdesign': None,
        'inside_medium': 'wet_gas_mixture', 
        'inside_m_dot_kg_h': 28_000.0, 'inside_T_in_C': 300.0, 'inside_T_out_C': None,
        'outside_medium': 'dry_air',
        'outside_m_dot_kg_h': 21_000.0, 'outside_T_in_C': -15.0, 'outside_T_out_C': 145.5,
        'Q_kW': None, 'effectiveness': None,
    },
    # {
    #     'case': 'S01', 'mode': 'simulate', 'overdesign': 0.15,
    #     'inside_medium': 'dry_air', 
    #     'inside_m_dot_kg_h': 18_220.0, 'inside_T_in_C': 30.0, 'inside_T_out_C': None,
    #     'outside_medium': 'wet_gas_mixture',
    #     'outside_m_dot_kg_h': 28_380.0, 'outside_T_in_C': 400.0, 'outside_T_out_C': None,
    #     'Q_kW': None, 'effectiveness': None,
    # },
    # {
    #     'case': 'R01', 'mode': 'rate', 'overdesign': None,
    #     'inside_medium': 'dry_air', 
    #     'inside_m_dot_kg_h': 18_220.0, 'inside_T_in_C': 30.0, 'inside_T_out_C': None,
    #     'outside_medium': 'wet_gas_mixture',
    #     'outside_m_dot_kg_h': 28_380.0, 'outside_T_in_C': 400.0, 'outside_T_out_C': None,
    #     'Q_kW': None, 'effectiveness': 0.45,
    # },
]

pd.DataFrame(cases)


In [ ]:
def rating_kwargs(case):
    if case.get('Q_kW') is not None:
        return {'Q': case['Q_kW'] * 1e3}
    if case.get('effectiveness') is not None:
        return {'effectiveness': case['effectiveness']}
    return {}  # duty wynika z kompletnej strony bilansu


def provider_for_medium(medium_key):
    medium_cfg = media.get(medium_key)
    if medium_cfg is None or 'provider' not in medium_cfg:
        known = ', '.join(sorted(media.keys()))
        raise KeyError(f"Nieznane medium={medium_key!r}. Dostępne: {known}")
    return medium_cfg['provider']


process_rows, thermal_rows, fluid_rows, error_rows = [], [], [], []

for case in cases:
    try:
        inside_provider = provider_for_medium(case['inside_medium'])
        outside_provider = provider_for_medium(case['outside_medium'])

        mode = case['mode']
        common = {
            'case': case['case'], 'mode': mode,
            'euler_provider': euler_provider,
            'inside_medium': case['inside_medium'],
            'outside_medium': case['outside_medium'],
        }

        if mode == 'simulate':
            overdesign = case.get('overdesign', 0.0)
            if overdesign is None:
                overdesign = 0.0
            inside = HXSideInput(
                provider=inside_provider, p=101_325.0,
                m_dot=kgh_to_kgs(case['inside_m_dot_kg_h']),
                T_in=c_to_k(case['inside_T_in_C']),
            )
            outside = HXSideInput(
                provider=outside_provider, p=101_325.0,
                m_dot=kgh_to_kgs(case['outside_m_dot_kg_h']),
                T_in=c_to_k(case['outside_T_in_C']),
            )
            result = hx.simulate(
                inside, outside, surface_margin=overdesign,
                euler_provider=euler_provider,
            )
            warning_codes = ', '.join(w.code for w in (result.warnings or []))
            process_rows.append({
                **common,
                'inside_m_dot_kg_h': inside.m_dot * 3600.0,
                'inside_T_in_C': inside.T_in - 273.15,
                'inside_T_out_C': result.T_out_inside - 273.15,
                'outside_m_dot_kg_h': outside.m_dot * 3600.0,
                'outside_T_in_C': outside.T_in - 273.15,
                'outside_T_out_C': result.T_out_outside - 273.15,
                'inside_velocity_m_s': result.inside_velocity_mean,
                'outside_velocity_m_s': result.outside_velocity_mean,
                'inside_dp_total_Pa': result.final_result.tube_side_hydraulic.dp_total,
                'outside_dp_total_Pa': result.final_result.outside_side_hydraulic.dp_total,
                'Q_required_kW': None, 'Q_achievable_kW': result.q / 1e3,
                'effectiveness_required': None, 'warnings': warning_codes,
            })
            thermal_rows.append({
                **common,
                'inside_Re': result.inside_Re_mean,
                'outside_Re': result.outside_Re_mean,
                'inside_alfa_W_m2K': result.inside_alfa_mean,
                'outside_alfa_W_m2K': result.outside_alfa_mean,
                'U_mean_W_m2K': result.U_mean, 'UA_actual_W_K': result.UA,
                'UA_effective_W_K': result.UA / (1.0 + overdesign),
                'UA_required_W_K': None, 'A_actual_m2': result.final_result.A_o,
                'A_required_m2': None,
                'overdesign_input_pct': 100.0 * overdesign,
                'overdesign_pct': 100.0 * result.overdesign_factor,
                'UA_margin_pct': None,
            })
            fluid_rows.append({
                **common,
                'inside_rho_kg_m3': result.inside_props_mean.rho,
                'inside_mu_Pa_s': result.inside_props_mean.mu,
                'inside_k_W_mK': result.inside_props_mean.k,
                'inside_cp_J_kgK': result.inside_props_mean.cp,
                'outside_rho_kg_m3': result.outside_props_mean.rho,
                'outside_mu_Pa_s': result.outside_props_mean.mu,
                'outside_k_W_mK': result.outside_props_mean.k,
                'outside_cp_J_kgK': result.outside_props_mean.cp,
            })

        elif mode == 'rate':
            inside = BalanceSideSpec(
                provider=inside_provider, p=101_325.0,
                m_dot=kgh_to_kgs(case['inside_m_dot_kg_h']),
                T_in=c_to_k(case['inside_T_in_C']), T_out=c_to_k(case['inside_T_out_C']),
            )
            outside = BalanceSideSpec(
                provider=outside_provider, p=101_325.0,
                m_dot=kgh_to_kgs(case['outside_m_dot_kg_h']),
                T_in=c_to_k(case['outside_T_in_C']), T_out=c_to_k(case['outside_T_out_C']),
            )
            result = hx.rate(
                inside, outside, include_simulation=True,
                euler_provider=euler_provider, **rating_kwargs(case),
            )
            balance = result.closed_balance
            warning_codes = ', '.join(w.code for w in (result.warnings or []))

            simulation = result.simulation
            if simulation is not None:
                inside_re = simulation.inside_Re_mean
                outside_re = simulation.outside_Re_mean
                inside_alfa = simulation.inside_alfa_mean
                outside_alfa = simulation.outside_alfa_mean
                inside_velocity = simulation.inside_velocity_mean
                outside_velocity = simulation.outside_velocity_mean
                inside_dp = simulation.final_result.tube_side_hydraulic.dp_total
                outside_dp = simulation.final_result.outside_side_hydraulic.dp_total
                inside_rho = simulation.inside_props_mean.rho
                inside_mu = simulation.inside_props_mean.mu
                inside_k = simulation.inside_props_mean.k
                inside_cp = simulation.inside_props_mean.cp
                outside_rho = simulation.outside_props_mean.rho
                outside_mu = simulation.outside_props_mean.mu
                outside_k = simulation.outside_props_mean.k
                outside_cp = simulation.outside_props_mean.cp
            else:
                inside_re = float('nan')
                outside_re = float('nan')
                inside_alfa = float('nan')
                outside_alfa = float('nan')
                inside_velocity = float('nan')
                outside_velocity = float('nan')
                inside_dp = float('nan')
                outside_dp = float('nan')
                inside_rho = float('nan')
                inside_mu = float('nan')
                inside_k = float('nan')
                inside_cp = float('nan')
                outside_rho = float('nan')
                outside_mu = float('nan')
                outside_k = float('nan')
                outside_cp = float('nan')

            process_rows.append({
                **common,
                'inside_m_dot_kg_h': balance.inside.m_dot * 3600.0,
                'inside_T_in_C': balance.inside.T_in - 273.15,
                'inside_T_out_C': balance.inside.T_out - 273.15,
                'outside_m_dot_kg_h': balance.outside.m_dot * 3600.0,
                'outside_T_in_C': balance.outside.T_in - 273.15,
                'outside_T_out_C': balance.outside.T_out - 273.15,
                'inside_velocity_m_s': inside_velocity,
                'outside_velocity_m_s': outside_velocity,
                'inside_dp_total_Pa': inside_dp,
                'outside_dp_total_Pa': outside_dp,
                'Q_required_kW': result.Q_required / 1e3,
                'Q_achievable_kW': result.Q_achievable / 1e3,
                'effectiveness_required': balance.effectiveness,
                'warnings': warning_codes,
            })
            thermal_rows.append({
                **common,
                'inside_Re': inside_re,
                'outside_Re': outside_re,
                'inside_alfa_W_m2K': inside_alfa,
                'outside_alfa_W_m2K': outside_alfa,
                'U_mean_W_m2K': result.U_mean, 'UA_actual_W_K': result.UA_actual,
                'UA_effective_W_K': None, 'UA_required_W_K': result.UA_required,
                'A_actual_m2': result.A_o, 'A_required_m2': result.A_required,
                'overdesign_input_pct': None,
                'overdesign_pct': 100.0 * result.overdesign_factor,
                'UA_margin_pct': 100.0 * result.ua_margin,
            })
            fluid_rows.append({
                **common,
                'inside_rho_kg_m3': inside_rho,
                'inside_mu_Pa_s': inside_mu,
                'inside_k_W_mK': inside_k,
                'inside_cp_J_kgK': inside_cp,
                'outside_rho_kg_m3': outside_rho,
                'outside_mu_Pa_s': outside_mu,
                'outside_k_W_mK': outside_k,
                'outside_cp_J_kgK': outside_cp,
            })
        else:
            raise ValueError(f"Nieobsługiwany mode={mode!r}; użyj 'simulate' lub 'rate'.")
    except Exception as exc:
        error_rows.append({
            'case': case['case'], 'mode': case['mode'],
            'euler_provider': euler_provider, 'error': str(exc),
        })

process_results = pd.DataFrame(process_rows)
thermal_results = pd.DataFrame(thermal_rows)
fluid_results = pd.DataFrame(fluid_rows)
errors = pd.DataFrame(error_rows)
assert errors.empty, errors.to_dict('records')
assert len(process_results) == len(cases) == len(thermal_results) == len(fluid_results)

## Thermal process results

In [ ]:
process_table = process_results.round({
    'inside_m_dot_kg_h': 1, 'inside_T_in_C': 2, 'inside_T_out_C': 2,
    'outside_m_dot_kg_h': 1, 'outside_T_in_C': 2, 'outside_T_out_C': 2,
    'inside_velocity_m_s': 3, 'outside_velocity_m_s': 3,
    'inside_dp_total_Pa': 1, 'outside_dp_total_Pa': 1,
    'Q_required_kW': 2, 'Q_achievable_kW': 2, 'effectiveness_required': 4,
})
thermal_table = thermal_results.round({
    'inside_Re': 0, 'outside_Re': 0,
    'inside_alfa_W_m2K': 2, 'outside_alfa_W_m2K': 2,
    'U_mean_W_m2K': 2, 'UA_actual_W_K': 1, 'UA_effective_W_K': 1,
    'UA_required_W_K': 1,
    'A_actual_m2': 2, 'A_required_m2': 2,
    'overdesign_input_pct': 2, 'overdesign_pct': 2, 'UA_margin_pct': 2,
})
fluid_table = fluid_results.round({
    'inside_rho_kg_m3': 4, 'outside_rho_kg_m3': 4,
    'inside_mu_Pa_s': 8, 'outside_mu_Pa_s': 8,
    'inside_k_W_mK': 5, 'outside_k_W_mK': 5,
    'inside_cp_J_kgK': 2, 'outside_cp_J_kgK': 2,
})

export_dir = workspace_root / 'core' / 'tests' / 'outputs'
export_dir.mkdir(parents=True, exist_ok=True)
export_path = export_dir / 'heat_balance_rating_matrix_results.xlsx'

with pd.ExcelWriter(export_path) as writer:
    process_table.to_excel(writer, sheet_name='process', index=False)
    thermal_table.to_excel(writer, sheet_name='thermal', index=False)
    fluid_table.to_excel(writer, sheet_name='fluid_props', index=False)

display(process_table)
display(thermal_table)
display(fluid_table)
print(f'Zapisano plik Excel: {export_path}')
